## Making a single data frame

In [3]:
import pandas as pd


In [4]:
# I am combining three csv files, each with thesis titles 
#that contain the keywords 'Cochin', 'Kochi', and 'ernakulam to make a master csv named "thesis_titles_master"
df_1 = pd.read_csv('cochin_thesis_titles.csv')
df_2 = pd.read_csv('kochi_thesis_titles.csv')
df_3 = pd.read_csv('ernakulam_thesis_titles.csv')

df = pd.concat([df_1,df_2,df_3], ignore_index = True)

In [5]:
df.shape

(351, 4)

In [6]:
df.columns

Index(['Upload Date', 'Title', 'Researcher', 'Guide(s)'], dtype='str')

In [7]:
df.to_csv("thesis_titles_master.csv", index = False)

In [8]:
#Here I am making a mini data frame with just the 'Title' column and 
#saving this data frame as a csv for further analysis
df_titles = df.drop(columns =['Upload Date', 'Researcher', 'Guide(s)'])

In [9]:
df_titles.to_csv("thesis_titles.csv", index = False)

## Making new dataframes after handcoding subordinate categories and ai-coded superordinate categories

In [2]:
df_classified = pd.read_csv('thesis_titles_broad_categories_no_duplicates.csv')

In [3]:
df_classified.columns

Index(['Title', 'category'], dtype='str')

In [5]:
categories = df_classified.sort_values('category')

In [7]:
categories.to_csv("titles_by_categories.csv", index = False)

In [6]:
#df_classified.shape

In [30]:
df_counts = pd.read_csv('ThesisCategorisation_Full.csv')
df_counts.shape

(347, 3)

In [32]:
df_counts.head()

,Title,category,parent_category
0,Descriptive study of Cochin Konkani,art and culture,Art and Cultural Memory
1,Exploring the right to the city an enquiry int...,art and culture,Art and Cultural Memory
2,Wide Angle Lens as a Flaneur in Imagining the ...,art and culture,Art and Cultural Memory
3,A Study of Temples in Ernakulam and Thrissur D...,art and culture,Art and Cultural Memory
4,An Analysis On The Service Quality Of Organise...,business,Industry and Economy


## Percentages and Counts of subordinate and parent categories

In [34]:
#workflow to add value counts as a percentage to a separate column named percentage. and round of the % to 2 decimal places. 
# Caveat: if '%' symbol is attached here, tools like datawrapper will read it as Text/String column. 
# instead of counts, I should have used the word percentages!
parent_category_counts = df_counts['parent_category'].value_counts(normalize = True).to_frame()
parent_category_counts.columns = ['percentage']
parent_category_counts['percentage']=(parent_category_counts['percentage']*100).round(2)
parent_category_counts

,percentage
parent_category,
Science and Environment,41.21
"Health, Society and Wellbeing",24.21
Industry and Economy,23.05
Education,6.63
Art and Cultural Memory,4.90


In [29]:
parent_category_counts.to_csv('parent_category_percentages.csv')

In [35]:
parent_category_numbers = df_counts['parent_category'].value_counts()
parent_category_numbers

parent_category
Science and Environment          143
Health, Society and Wellbeing     84
Industry and Economy              80
Education                         23
Art and Cultural Memory           17
Name: count, dtype: int64

In [36]:
parent_category_numbers.to_csv('parent_category_numbers.csv')

In [54]:
subordinate_category_numbers = df_counts['category'].value_counts()
subordinate_category_numbers


category
ecology/aquatic/scientific    136
socioeconomics                 36
business                       34
health                         30
business-hr                    19
history/socioeconomics         18
social issues                  16
education and literacy         14
history                        10
higher education                9
waste management                7
business-cochin port            6
art and culture                 4
business-sez                    3
media impact                    3
culture                         2
Name: count, dtype: int64

In [42]:
subordinate_category_numbers.to_csv('sub_category_numbers.csv')

In [15]:
subordinate_category_counts = df_counts['category'].value_counts(normalize = True).to_frame()
subordinate_category_counts.columns = ['percentage']
subordinate_category_counts['percentage']=(subordinate_category_counts['percentage']*100).round(2)
subordinate_category_counts

,percentage
category,
ecology/aquatic/scientific,39.19
socioeconomics,10.37
business,9.80
health,8.65
business-hr,5.48
history/socioeconomics,5.19
social issues,4.61
education and literacy,4.03
history,2.88


In [50]:
subordinate_category_counts.to_csv('subordinate_category_percentages.csv')

## Charting the top 2 parent categories as diverging intellectual forces 

In [20]:
df_counts.columns    
    

Index(['Title', 'category', 'parent_category'], dtype='str')

In [38]:
# Culling the top 3 : Science and Environment-41.21, Health, Society and Wellbeing-24.21, Industry and Economy-23.05
# in the parent_category_counts dataframe by locating the row label (e.g. Science and Environment) and column label (perecentage)
# I am also aggregating the percentage of 'Health, Society, Wellbeing', and 'Industry and Economy'.
science_pct = parent_category_counts.loc['Science and Environment', 'percentage']
society_and_economy_pct = (parent_category_counts.loc['Health, Society and Wellbeing', 'percentage'] + 
                           parent_category_counts.loc['Industry and Economy','percentage'])


In [47]:
divergence_percentage_data = {
    'domain':['% of PhD studies'],
    'Science and Environment': [science_pct],
    'Industry and Economy + Health, Society, and Wellbeing': [society_and_economy_pct]
}

In [48]:
df_divergence_data = pd.DataFrame(divergence_percentage_data)
df_divergence_data = df_divergence_data.round(2)
df_divergence_data.to_csv('PhD studies divergence.csv', index = False)

## Charting the relationship of subordinate categories with parent categories - RAWGraph Chart

Science and Environment          
Health, Society and Wellbeing     
Industry and Economy              
Education                         
Art and Cultural Memory  

ecology/aquatic/scientific   
socioeconomics                 
business                       
health                         
business-hr                    
history/socioeconomics         
social issues                  
education and literacy         
history                        
higher education               
waste management                
business-cochin port            
art and culture                 
business-sez                    
media impact                    
culture                         

In [55]:
# Making a new dataframe that has information grouped by parent_categories and categories.(no thesis titles). 

hierarchy = df_counts.groupby(['parent_category', 'category']).size().reset_index(name='count_of_thesis')

#Here I am sorting the hierarchy data frame so that the parent categories are arranged in reverse alphabetical order (descending). 
#I am doing this becuase I want the parent and sub categories with maximum counts (Science and Environment : ecology/aquatic/scientific ) 
#in the top of the csv file(False for 'parent_category', and False for 'count_of_thesis']
  
hierarchy = hierarchy.sort_values(by=['parent_category', 'count_of_thesis'], ascending=[False, False])

In [56]:
hierarchy.to_csv("hierarchy_rawgraph.csv", index = False)

## Charting in D3

In [2]:
import json
import pandas as pd

# 1. Load your uploaded dataset
df = pd.read_csv("hierarchy_rawgraph.csv")

# 2. Build the root of the tree
tree = {"name": "Kochi Theses", "children": []}

# 3. Group by parent category dynamically
for parent, parent_group in df.groupby("parent_category"):
    parent_node = {"name": parent, "children": []}

    # 4. Add each subordinate category inside this parent branch
    for _, row in parent_group.iterrows():
        child_node = {
            "name": row["category"],
            "value": int(row["count_of_thesis"]),  # Matches your file's column header!
        }
        parent_node["children"].append(child_node)

    tree["children"].append(parent_node)

# 5. Export directly as a JSON file for D3
with open("kochi_tree.json", "w") as f:
    json.dump(tree, f, indent=2)

print("✅ Data transformed successfully using hierarchy_rawgraph.csv!")

✅ Data transformed successfully using hierarchy_rawgraph.csv!
